In [124]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

In [125]:
df = pd.read_excel("spam.xlsx")

In [126]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [127]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   str   
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     str   
 3   Unnamed: 3  12 non-null     str   
 4   Unnamed: 4  6 non-null      str   
dtypes: object(1), str(4)
memory usage: 239.9+ KB


In [128]:
df = df.drop( columns= ['Unnamed: 2',	'Unnamed: 3'	,'Unnamed: 4'])

In [129]:
df= df.rename( columns={'v1':'label', 'v2':'text'})

In [130]:
df['text']

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                Will �_ b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: text, Length: 5572, dtype: object

In [131]:
df['text']= df['text'].astype(str)
df_clean = df.copy()
df_clean['text'] = df['text'].str.lower()

In [132]:
df_clean['text'].dtype

<StringDtype(na_value=nan)>

In [133]:
df_clean['text'].dropna(inplace= True)

In [134]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to C:\Users\Syam
[nltk_data]     Sundar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Syam
[nltk_data]     Sundar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Syam
[nltk_data]     Sundar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [135]:
stop_words = set( stopwords.words('english'))

In [136]:
def remove_puc(msg):
    clean_text = re.sub(r'[^\w\s]', '', msg)
    clean_text
    return clean_text

In [137]:
 df_clean['text'] = df_clean['text'].apply(remove_puc)

In [138]:
from nltk.tokenize import word_tokenize

lemmetizer = WordNetLemmatizer()
df_clean['text']  = df_clean['text'].apply(word_tokenize)


In [139]:
#  Remove stop words 
df_clean['text'] = df_clean['text'].apply( lambda val : [words for  words in val  if words not in stop_words] )
df_clean['text'] = df_clean['text'].apply( lambda val: [lemmetizer.lemmatize(words) for words in val] )


In [144]:
df_clean['text'] = df_clean['text'].apply( lambda val: " ".join([words for words in val]) )

In [145]:
df_clean['text']

0       go jurong point crazy available bugis n great ...
1                                 ok lar joking wif u oni
2       free entry 2 wkly comp win fa cup final tkts 2...
3                     u dun say early hor u c already say
4                nah dont think go usf life around though
                              ...                        
5567    2nd time tried 2 contact u u 750 pound prize 2...
5568                          _ b going esplanade fr home
5569                           pity mood soany suggestion
5570    guy bitching acted like id interested buying s...
5571                                       rofl true name
Name: text, Length: 5572, dtype: str

In [149]:
df_clean['label'] = df_clean['label'].map({ 'ham':0, 'spam':1}) 

In [152]:
label = df_clean['label']

In [168]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report , f1_score, precision_score, accuracy_score, confusion_matrix

In [153]:
vectorizer = TfidfVectorizer(ngram_range=(1,2) )
X = vectorizer.fit_transform(df['text'])
model = LogisticRegression()

X_train, X_test, y_train, y_test =  train_test_split(X, label , test_size= 0.3 , random_state= 67)





In [155]:
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [157]:
y_predict =  model.predict(X_test )

In [158]:
y_predict

array([0, 0, 0, ..., 0, 0, 0])

In [164]:
precision = precision_score(y_test, y_predict )
accuracy = accuracy_score(y_test, y_predict )
precision
accuracy

0.9300239234449761

In [167]:
report = classification_report( y_test, y_predict )
print(report)

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1439
           1       1.00      0.50      0.66       233

    accuracy                           0.93      1672
   macro avg       0.96      0.75      0.81      1672
weighted avg       0.94      0.93      0.92      1672



In [169]:
confu = confusion_matrix(y_test, y_predict)
confu

array([[1439,    0],
       [ 117,  116]])

In [170]:
df_clean[df_clean['label'] ==1]


,label,text
2,1,free entry 2 wkly comp win fa cup final tkts 2...
5,1,freemsg hey darling 3 week word back id like f...
8,1,winner valued network customer selected receiv...
9,1,mobile 11 month u r entitled update latest col...
11,1,six chance win cash 100 20000 pound txt csh11 ...
...,...,...
5537,1,want explicit sex 30 sec ring 02073162414 cost...
5540,1,asked 3mobile 0870 chatlines inclu free min in...
5547,1,contract mobile 11 mnths latest motorola nokia...
5566,1,reminder o2 get 250 pound free call credit det...


In [171]:
df[df['label'] =='spam']

,label,text
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
5,spam,FreeMsg Hey there darling it's been 3 week's n...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...
11,spam,"SIX chances to win CASH! From 100 to 20,000 po..."
...,...,...
5537,spam,Want explicit SEX in 30 secs? Ring 02073162414...
5540,spam,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,spam,Had your contract mobile 11 Mnths? Latest Moto...
5566,spam,REMINDER FROM O2: To get 2.50 pounds free call...


In [175]:
y_prob = model.predict_proba(X_test)[:, 1]
threshold = 0.2
y_pred_new = (y_prob >= threshold).astype(int)

print(f"\nThreshold: {threshold}")
print(classification_report(y_test, y_pred_new))
print( confusion_matrix(y_test, y_pred_new))


Threshold: 0.2
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1439
           1       0.97      0.86      0.91       233

    accuracy                           0.98      1672
   macro avg       0.97      0.93      0.95      1672
weighted avg       0.98      0.98      0.98      1672

[[1432    7]
 [  33  200]]


In [179]:
import pickle as pkl

#  Data_x , Data_y , tfidf , 


pkl.dump(vectorizer , open('tfidf.pkl', 'wb') )
pkl.dump(model , open('model.pkl', 'wb') )


